# Visualización de mapas de atención

Análisis cualitativo de qué regiones de la imagen priorizan los modelos entrenados. Se usa attention rollout para DeiT y GradCAM para MobileViT.

## Setup (Colab)

In [ ]:
# !pip install -q -r requirements.txt

## Attention rollout — DeiT-tiny

In [ ]:
# pendiente

## GradCAM — MobileViT-small

In [ ]:
# pendiente

## Comparación visual: ¿el modelo mira donde corresponde?

In [ ]:
# pendiente

In [ ]:
from pathlib import Path
import os
import json
from huggingface_hub import snapshot_download
import fiftyone as fo
from fiftyone import ViewField as F

REPO_ID = "harpreetsahota/CarDD"
REPO_TYPE = "dataset"

EXPORT_DIR = Path("../data/interim/cardd").resolve()
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

base_dir_env = os.environ.get("CARDD_SNAPSHOT_DIR")
if base_dir_env:
    BASE_DIR = Path(base_dir_env).expanduser().resolve()
    print(f"[INFO] Using CARDD_SNAPSHOT_DIR from env: {BASE_DIR}")
else:
    BASE_DIR = Path(snapshot_download(repo_id=REPO_ID, repo_type=REPO_TYPE))
    print(f"[INFO] Using HF cached snapshot: {BASE_DIR}")

required = ["samples.json", "metadata.json", "fiftyone.yml"]
missing = [f for f in required if not (BASE_DIR / f).exists()]
assert not missing, f"Missing required files {missing} at: {BASE_DIR}"

print("Using BASE_DIR:", BASE_DIR)
print("Top-level entries:", [p.name for p in sorted(BASE_DIR.iterdir())[:30]])

dataset_name = "cardd_local"



if dataset_name in fo.list_datasets():
    fo.delete_dataset(dataset_name)

ds = fo.Dataset.from_dir(
    dataset_dir=str(BASE_DIR),
    dataset_type=fo.types.FiftyOneDataset,
    name=dataset_name,
)

print(ds)
schema = ds.get_field_schema()
print("Fields:", list(schema.keys()))
candidate_fields = []
for field_name in schema.keys():
    if field_name in ("id", "filepath", "tags", "metadata", "created_at", "last_modified_at"):
        continue
    candidate_fields.append(field_name)

print("Candidate label fields:", candidate_fields)

# Elegir explícitamente el primer campo de labels razonable
LABEL_FIELD = None
for field_name in candidate_fields:
    sample = ds.first()
    value = sample.get(field_name, None)
    if value is not None:
        LABEL_FIELD = field_name
        break

assert LABEL_FIELD is not None, "No label field found"
print("Using LABEL_FIELD:", LABEL_FIELD)

# Ver una muestra real del campo
sample0 = ds.first()
print("Sample label object:", sample0[LABEL_FIELD])

# Si el campo tiene atributo .detections, filtramos no vacíos
value0 = sample0[LABEL_FIELD]
if hasattr(value0, "detections"):
    view = ds.match(F(f"{LABEL_FIELD}.detections").length() > 0)
    print("Samples with instances:", len(view))
else:
    view = ds
    print(f"[WARN] {LABEL_FIELD} does not expose .detections directly; using full dataset for now")

print("Ready for next step: inspect labels and export to COCO")

Fetching ... files: 2823it [00:00, 78648.42it/s]

[INFO] Using HF cached snapshot: /home/pablo/.cache/huggingface/hub/datasets--harpreetsahota--CarDD/snapshots/56900bde8dddfe00eb7c03114a1d46e9105e3cdb
Using BASE_DIR: /home/pablo/.cache/huggingface/hub/datasets--harpreetsahota--CarDD/snapshots/56900bde8dddfe00eb7c03114a1d46e9105e3cdb
Top-level entries: ['.gitattributes', 'CarDD_license.pdf', 'README.md', 'cardd-overview-lq.gif', 'data', 'fiftyone.yml', 'metadata.json', 'samples.json']
Using FiftyOne config: /home/pablo/repos/car-damage-vit/notebooks/fiftyone_config.json


FiftyOneConfigError: MongoDB could not be installed on your system. Please define a `database_uri` in your `fiftyone.core.config.FiftyOneConfig` to connect to yourown MongoDB instance or cluster 